### Проект по созданию модели для ранжирования разработчиков на выполнение задач на основе их исторической активности

Источники:
1. Данные по Jira-задачам (data/raw/jira_issues.csv)
2. Данные по коммитам в GitLab (data/raw/gitlab_commits.csv)
3. Данные по сотрудникам (data/raw/employees.csv)

## 0. Настройка окружения

In [ ]:
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
from src.config import RANDOM_SEED
from src.data.cleaner import clean_data
from src.data.loader import load_data
from src.features.profiles import build_developer_profiles
from src.features.split import create_train_val_test_split

print("Импорты готовы")

## 1. Загрузка данных

In [ ]:
employees, jira, gitlab = load_data("../data/raw")

# Базовая информация
print("=== Jira Issues ===")
print(f"Размер: {jira.shape}")
print(jira.info())
print(jira.head())
print()

print("=== GitLab Commits ===")
print(f"Размер: {gitlab.shape}")
print(gitlab.info())
print(gitlab.head())
print()

print("=== Employees ===")
print(f"Размер: {employees.shape}")
print(employees.info())
print(employees.head())

## 2. Разведочный анализ (EDA)

In [ ]:
# Проверка пропусков
print("=== Пропуски в данных ===")

print("\nJira:")
print(jira.isnull().sum())

print("\nGitLab:")
print(gitlab.isnull().sum())

print("\nEmployees:")
print(employees.isnull().sum())

In [ ]:
# Базовая статистика
print("=== Статистика ===")

print("\nJira:")
print(jira.describe())

print("\nGitLab:")
print(gitlab.describe())

print("\nEmployees:")
print(employees.describe())

In [ ]:
# Pie chart для position_nm (топ-10 + остальные)
position_counts = employees["position_nm"].value_counts()
top_10 = position_counts.head(10)
others_sum = position_counts.iloc[10:].sum()

if others_sum > 0:
    top_10["Остальные"] = others_sum

plt.figure(figsize=(12, 10))
plt.pie(top_10.values, labels=top_10.index, autopct="%1.1f%%", startangle=90)
plt.title("Распределение по позициям (Топ-10 + Остальные)")
plt.axis("equal")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: опыт работы vs количество коммитов
gitlab_commits = gitlab.groupby("author_login").size().reset_index(name="commits_count")
emp_gitlab = employees.merge(gitlab_commits, left_on="login", right_on="author_login", how="left")
emp_gitlab["commits_count"] = emp_gitlab["commits_count"].fillna(0)
emp_gitlab = emp_gitlab[emp_gitlab["commits_count"] > 0]

plt.figure(figsize=(10, 6))
plt.scatter(emp_gitlab["work_experience_day_cnt"], emp_gitlab["commits_count"], alpha=0.5)
plt.xlabel("Опыт работы (дни)")
plt.ylabel("Количество коммитов")
plt.title("Опыт работы vs Количество коммитов")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Scatter plot: опыт работы vs количество Jira задач
jira_tasks = jira.groupby("assignee_login_nm").size().reset_index(name="tasks_count")
emp_jira = employees.merge(jira_tasks, left_on="login", right_on="assignee_login_nm", how="left")
emp_jira["tasks_count"] = emp_jira["tasks_count"].fillna(0)
emp_jira = emp_jira[emp_jira["tasks_count"] > 0]

plt.figure(figsize=(10, 6))
plt.scatter(emp_jira["work_experience_day_cnt"], emp_jira["tasks_count"], alpha=0.5, color="orange")
plt.xlabel("Опыт работы (дни)")
plt.ylabel("Количество Jira задач")
plt.title("Опыт работы vs Количество Jira задач")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Очистка данных

In [ ]:
# Полный пайплайн очистки
print(f"До очистки: Employees={employees.shape[0]}, Jira={jira.shape[0]}, GitLab={gitlab.shape[0]}")

employees, jira, gitlab = clean_data(employees, jira, gitlab)

print(
    f"После очистки: Employees={employees.shape[0]}, Jira={jira.shape[0]}, GitLab={gitlab.shape[0]}"
)

In [ ]:
print("=== Проверка дубликатов ===")

print(
    f"Jira по (assignee_login_nm, issue_rk): {jira.duplicated(subset=['assignee_login_nm', 'issue_rk']).sum()}"
)
print(f"GitLab по commit_hash: {gitlab.duplicated(subset=['commit_hash']).sum()}")
print(f"Employees по login: {employees.duplicated(subset=['login']).sum()}")

## 4. Проверка связей между таблицами

In [ ]:
print("=== Проверка связей между таблицами ===")

jira_logins = set(jira["assignee_login_nm"].unique())
gitlab_logins = set(gitlab["author_login"].unique())
emp_logins = set(employees["login"].unique())

print(f"\nJira исполнителей нет в employees: {len(jira_logins - emp_logins)}")
print(f"GitLab авторов нет в employees: {len(gitlab_logins - emp_logins)}")
print(f"\nСотрудников без Jira задач: {len(emp_logins - jira_logins)}")
print(f"Сотрудников без GitLab коммитов: {len(emp_logins - gitlab_logins)}")

## 5. Train/Val/Test Split

In [ ]:
# Разделение на train/val/test
jira_train, jira_val, jira_test = create_train_val_test_split(
    jira, test_size=0.2, val_size=0.2, random_state=RANDOM_SEED
)

print(f"Train: {len(jira_train)} ({len(jira_train) / len(jira) * 100:.1f}%)")
print(f"Val: {len(jira_val)} ({len(jira_val) / len(jira) * 100:.1f}%)")
print(f"Test: {len(jira_test)} ({len(jira_test) / len(jira) * 100:.1f}%)")

## 6. Профили разработчиков

In [ ]:
# Построение профилей
dev_profiles = build_developer_profiles(employees, jira, gitlab, jira_train=jira_train)

print(f"Размер профилей: {dev_profiles.shape[0]} сотрудников")
print(
    dev_profiles[
        ["login", "position_nm", "jira_issues_count", "commits_count", "projects_count", "profile_words_count"]
    ].head()
)

## 7. Исследование выбросов

In [ ]:
# Employees: Анализ выбросов
print("=== Employees: Анализ выбросов ===")
print("work_experience_day_cnt:")
print(employees["work_experience_day_cnt"].describe())
print(f"Мин: {employees['work_experience_day_cnt'].min()} дней")
print(f"Макс: {employees['work_experience_day_cnt'].max()} дней")

Q1 = employees["work_experience_day_cnt"].quantile(0.25)
Q3 = employees["work_experience_day_cnt"].quantile(0.75)
IQR = Q3 - Q1
print(f"IQR: {IQR} (Q1={Q1}, Q3={Q3})")

outliers_low = (employees["work_experience_day_cnt"] < Q1 - 1.5 * IQR).sum()
outliers_high = (employees["work_experience_day_cnt"] > Q3 + 1.5 * IQR).sum()
print(f"Выбросов: низких={outliers_low}, высоких={outliers_high}")

In [ ]:
# Визуализация выбросов
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].boxplot(employees["work_experience_day_cnt"], vert=True)
axes[0].set_title("Опыт работы (дни)")
axes[1].boxplot(jira["issue_desc_clean"].str.len(), vert=True)
axes[1].set_title("Длина описания Jira")
axes[2].boxplot(gitlab["commit_title_clean"].str.len(), vert=True)
axes[2].set_title("Длина заголовка GitLab")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7.1. Обработка выбросов (IQR-метод)

In [ ]:
print("=== Обработка выбросов (IQR-метод) ===")

print("\n--- Валидация ---")
print(f"До валидации: Employees={employees.shape[0]}, GitLab={gitlab.shape[0]}")

# Опыт работы < 1 дня (невозможен)
employees = employees[employees["work_experience_day_cnt"] >= 1]

# Мусорные заголовки коммитов
trash_titles = ["wip", "fix", "upd", "test", "tmp", "merge", "revert"]
gitlab = gitlab[~gitlab["commit_title_clean"].str.lower().isin(trash_titles)]

print(f"После валидации: Employees={employees.shape[0]}, GitLab={gitlab.shape[0]}")


print("\n--- IQR-обработка: Опыт работы ---")
Q1_exp = employees["work_experience_day_cnt"].quantile(0.25)
Q3_exp = employees["work_experience_day_cnt"].quantile(0.75)
IQR_exp = Q3_exp - Q1_exp
lower_bound = Q1_exp - 1.5 * IQR_exp
upper_bound = Q3_exp + 1.5 * IQR_exp

print(f"IQR={IQR_exp:.0f}, границы=[{lower_bound:.0f}, {upper_bound:.0f}]")
print(f"До: {employees.shape[0]} сотрудников")

employees_before = employees.shape[0]
employees = employees[
    (employees["work_experience_day_cnt"] >= max(1, lower_bound))
    & (employees["work_experience_day_cnt"] <= upper_bound)
]
print(
    f"После: {employees.shape[0]} сотрудников (удалено {employees_before - employees.shape[0]} выбросов)"
)


print("\n--- IQR-обработка: Длина описания Jira ---")
jira["issue_desc_len"] = jira["issue_desc_clean"].str.len()
Q1_desc = jira["issue_desc_len"].quantile(0.25)
Q3_desc = jira["issue_desc_len"].quantile(0.75)
IQR_desc = Q3_desc - Q1_desc
lower_desc = max(0, Q1_desc - 1.5 * IQR_desc)
upper_desc = Q3_desc + 1.5 * IQR_desc

print(f"IQR={IQR_desc:.0f}, границы=[{lower_desc:.0f}, {upper_desc:.0f}]")
print(f"До: {jira.shape[0]} задач")

jira_before = jira.shape[0]
jira = jira[(jira["issue_desc_len"] >= lower_desc) & (jira["issue_desc_len"] <= upper_desc)]
print(f"После: {jira.shape[0]} задач (удалено {jira_before - jira.shape[0]} выбросов)")


print("\n--- IQR-обработка: Длина заголовка GitLab ---")
gitlab["commit_title_len"] = gitlab["commit_title_clean"].str.len()
Q1_title = gitlab["commit_title_len"].quantile(0.25)
Q3_title = gitlab["commit_title_len"].quantile(0.75)
IQR_title = Q3_title - Q1_title
lower_title = max(0, Q1_title - 1.5 * IQR_title)
upper_title = Q3_title + 1.5 * IQR_title

print(f"IQR={IQR_title:.0f}, границы=[{lower_title:.0f}, {upper_title:.0f}]")
print(f"До: {gitlab.shape[0]} коммитов")

gitlab_before = gitlab.shape[0]
gitlab = gitlab[
    (gitlab["commit_title_len"] >= lower_title) & (gitlab["commit_title_len"] <= upper_title)
]
print(f"После: {gitlab.shape[0]} коммитов (удалено {gitlab_before - gitlab.shape[0]} выбросов)")

print("\n=== Итого после обработки выбросов ===")
print(f"Employees: {employees.shape[0]}")
print(f"Jira: {jira.shape[0]}")
print(f"GitLab: {gitlab.shape[0]}")

## 8. Сохранение обработанных данных

In [ ]:
# Сохранение очищенных данных
employees.to_csv("../data/processed/employees_clean.csv", index=False)
jira.to_csv("../data/processed/jira_clean.csv", index=False)
gitlab.to_csv("../data/processed/gitlab_clean.csv", index=False)

# Сохранение сплитов
jira_train.to_csv("../data/processed/jira_train.csv", index=False)
jira_val.to_csv("../data/processed/jira_val.csv", index=False)
jira_test.to_csv("../data/processed/jira_test.csv", index=False)

# Сохранение профилей
dev_profiles.to_csv("../data/processed/dev_profiles.csv", index=False)

print("Данные сохранены в data/processed/")
print("  - employees_clean.csv")
print("  - jira_clean.csv")
print("  - gitlab_clean.csv")
print("  - jira_train.csv")
print("  - jira_val.csv")
print("  - jira_test.csv")
print("  - dev_profiles.csv (построены на train данных)")